# Warsaw parking dataset — workbench

Data: `silver/parking_dataset.csv` — one row per parking per snapshot, all columns flat.

To refresh: run `ping/ping_parkings.py` (new snapshot), then `transform/build_silver.py` (rebuild CSV), then re-run the cell below.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT = Path.cwd().parent if Path.cwd().name == "eda" else Path.cwd()

df = pd.read_csv(PROJECT / "silver" / "parking_dataset.csv",
                 parse_dates=["source_ts", "ingested_at"])
# note: source_ts is naive Warsaw local time; ingested_at is UTC

print(df.shape)

(98, 59)


In [2]:
df.head()

,parking_id,name,source_ts,ingested_at,free_public,free_disabled,free_electric,total_standard,total_disabled,total_electric,...,price_motorcycle_2h,price_motorcycle_3h,price_motorcycle_4h,price_van_0h,price_van_1h,price_van_24h,price_van_2h,price_van_3h,price_van_4h,security
0,parking_na_pl._żelaznej_bramy,Parking na Pl. Żelaznej Bramy,2026-07-10 12:21:05,2026-07-10 10:21:29.327023+00:00,12,0,0,47,0,0,...,11.0,14.0,18.0,NaN,6.0,102.0,13.0,18.0,22.0,Parking strzeżony
1,parking_pałac_kultury_i_nauki,Parking Pałac Kultury i Nauki,2026-07-10 12:21:05,2026-07-10 10:21:29.327023+00:00,252,0,0,603,22,1,...,NaN,NaN,NaN,10.0,30.0,450.0,NaN,NaN,NaN,NaN
2,parking_pl._powstańców_warszawy,Parking Pl. Powstańców Warszawy,2026-07-10 12:21:05,2026-07-10 10:21:29.327023+00:00,297,15,7,396,16,8,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,parking_pod_placem_krasińskich,PARKING POD PLACEM KRASIŃSKICH,2026-07-10 12:21:05,2026-07-10 10:21:29.327023+00:00,96,0,0,356,4,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,parking_pod_ulicą_waryńskiego,PARKING POD ULICĄ WARYŃSKIEGO,2026-07-10 12:21:05,2026-07-10 10:21:29.327023+00:00,21,0,0,130,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
pd.set_option('display.max_columns', None)

# Set pandas to display all rows (careful if the dataset is huge, use head())
# pd.set_option('display.max_rows', None) 

# Now view the data
df.head()

,parking_id,name,source_ts,ingested_at,free_public,free_disabled,free_electric,total_standard,total_disabled,total_electric,address,allowed_vehicles,category,latitude,levels,longitude,manager,manager_email,manager_phone,max_autobus_length_m,max_autobus_width_m,max_car_length_m,max_car_width_m,max_samochód_ciężarowy_length_m,max_samochód_ciężarowy_width_m,max_van_length_m,max_van_width_m,open_friday,open_monday,open_saturday,open_sunday,open_thursday,open_tuesday,open_wednesday,operator_phone,payment_methods,price_autobus_1h,price_autobus_24h,price_autobus_2h,price_autobus_3h,price_autobus_4h,price_car_0h,price_car_1h,price_car_24h,price_car_2h,price_car_3h,price_car_4h,price_motorcycle_1h,price_motorcycle_24h,price_motorcycle_2h,price_motorcycle_3h,price_motorcycle_4h,price_van_0h,price_van_1h,price_van_24h,price_van_2h,price_van_3h,price_van_4h,security
0,parking_na_pl._żelaznej_bramy,Parking na Pl. Żelaznej Bramy,2026-07-10 12:21:05,2026-07-10 10:21:29.327023+00:00,12,0,0,47,0,0,NaN,"samochód osobowy, samochód dostawczy, motor",Publiczny,52.239560,1,21.001779,Zarząd Terenów Publicznych,nopar@wp.pl,+48 601-248-664,NaN,NaN,5.0,2.5,NaN,NaN,5.0,2.5,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,+48 730-255-220,"Gotówka, karta płatnicza",NaN,NaN,NaN,NaN,NaN,NaN,5.0,98.0,11.0,14.0,18.0,5.0,98.0,11.0,14.0,18.0,NaN,6.0,102.0,13.0,18.0,22.0,Parking strzeżony
1,parking_pałac_kultury_i_nauki,Parking Pałac Kultury i Nauki,2026-07-10 12:21:05,2026-07-10 10:21:29.327023+00:00,252,0,0,603,22,1,"Warszawa, Plac Defilad 1","samochód osobowy, autokar, Samochód dostawczy",NaN,52.231693,1,21.008925,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,00:00-00:00,00:00-00:00,00:00-00:00,00:00-00:00,00:00-00:00,00:00-00:00,00:00-00:00,NaN,"Gotówka, karta płatnicza",NaN,NaN,NaN,NaN,NaN,6.0,8.0,120.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.0,30.0,450.0,NaN,NaN,NaN,NaN
2,parking_pl._powstańców_warszawy,Parking Pl. Powstańców Warszawy,2026-07-10 12:21:05,2026-07-10 10:21:29.327023+00:00,297,15,7,396,16,8,Pl. Powstańców Warszawy,samochód osobowy,NaN,52.234989,4,21.013279,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,parking_pod_placem_krasińskich,PARKING POD PLACEM KRASIŃSKICH,2026-07-10 12:21:05,2026-07-10 10:21:29.327023+00:00,96,0,0,356,4,0,NaN,"samochód osobowy, motor",Publiczny,52.249166,1,21.004752,City Parking Group S.A.,parking.krasinskich@citypg.pl,+48 785 604 276,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,+48 785 604 276,"Gotówka, karta płatnicza",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,parking_pod_ulicą_waryńskiego,PARKING POD ULICĄ WARYŃSKIEGO,2026-07-10 12:21:05,2026-07-10 10:21:29.327023+00:00,21,0,0,130,0,0,NaN,samochód osobowy,Publiczny,52.219031,1,21.015129,City Parking Group S.A.,parking.warynskiego@citypg.pl,+48 721 910 204,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,+48 721 910 204,"Gotówka, karta płatnicza",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
df.columns

Index(['parking_id', 'name', 'source_ts', 'ingested_at', 'free_public',
       'free_disabled', 'free_electric', 'total_standard', 'total_disabled',
       'total_electric', 'address', 'allowed_vehicles', 'category', 'latitude',
       'levels', 'longitude', 'manager', 'manager_email', 'manager_phone',
       'max_autobus_length_m', 'max_autobus_width_m', 'max_car_length_m',
       'max_car_width_m', 'max_samochód_ciężarowy_length_m',
       'max_samochód_ciężarowy_width_m', 'max_van_length_m', 'max_van_width_m',
       'open_friday', 'open_monday', 'open_saturday', 'open_sunday',
       'open_thursday', 'open_tuesday', 'open_wednesday', 'operator_phone',
       'payment_methods', 'price_autobus_1h', 'price_autobus_24h',
       'price_autobus_2h', 'price_autobus_3h', 'price_autobus_4h',
       'price_car_0h', 'price_car_1h', 'price_car_24h', 'price_car_2h',
       'price_car_3h', 'price_car_4h', 'price_motorcycle_1h',
       'price_motorcycle_24h', 'price_motorcycle_2h', 'price_motorcy

In [5]:
import folium

# latest row per parking (you now have multiple snapshots per parking)
latest = df.sort_values("source_ts").groupby("parking_id").tail(1)

m = folium.Map(location=[52.2319, 21.0067], zoom_start=13)  # Warsaw center

for _, p in latest.iterrows():
    folium.Marker(
        [p["latitude"], p["longitude"]],
        tooltip=p["name"],
        popup=f"{p['name']}<br>free: {p['free_public']} / {p['total_standard']}",
    ).add_to(m)



In [6]:
df.head()

,parking_id,name,source_ts,ingested_at,free_public,free_disabled,free_electric,total_standard,total_disabled,total_electric,address,allowed_vehicles,category,latitude,levels,longitude,manager,manager_email,manager_phone,max_autobus_length_m,max_autobus_width_m,max_car_length_m,max_car_width_m,max_samochód_ciężarowy_length_m,max_samochód_ciężarowy_width_m,max_van_length_m,max_van_width_m,open_friday,open_monday,open_saturday,open_sunday,open_thursday,open_tuesday,open_wednesday,operator_phone,payment_methods,price_autobus_1h,price_autobus_24h,price_autobus_2h,price_autobus_3h,price_autobus_4h,price_car_0h,price_car_1h,price_car_24h,price_car_2h,price_car_3h,price_car_4h,price_motorcycle_1h,price_motorcycle_24h,price_motorcycle_2h,price_motorcycle_3h,price_motorcycle_4h,price_van_0h,price_van_1h,price_van_24h,price_van_2h,price_van_3h,price_van_4h,security
0,parking_na_pl._żelaznej_bramy,Parking na Pl. Żelaznej Bramy,2026-07-10 12:21:05,2026-07-10 10:21:29.327023+00:00,12,0,0,47,0,0,NaN,"samochód osobowy, samochód dostawczy, motor",Publiczny,52.239560,1,21.001779,Zarząd Terenów Publicznych,nopar@wp.pl,+48 601-248-664,NaN,NaN,5.0,2.5,NaN,NaN,5.0,2.5,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,+48 730-255-220,"Gotówka, karta płatnicza",NaN,NaN,NaN,NaN,NaN,NaN,5.0,98.0,11.0,14.0,18.0,5.0,98.0,11.0,14.0,18.0,NaN,6.0,102.0,13.0,18.0,22.0,Parking strzeżony
1,parking_pałac_kultury_i_nauki,Parking Pałac Kultury i Nauki,2026-07-10 12:21:05,2026-07-10 10:21:29.327023+00:00,252,0,0,603,22,1,"Warszawa, Plac Defilad 1","samochód osobowy, autokar, Samochód dostawczy",NaN,52.231693,1,21.008925,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,00:00-00:00,00:00-00:00,00:00-00:00,00:00-00:00,00:00-00:00,00:00-00:00,00:00-00:00,NaN,"Gotówka, karta płatnicza",NaN,NaN,NaN,NaN,NaN,6.0,8.0,120.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.0,30.0,450.0,NaN,NaN,NaN,NaN
2,parking_pl._powstańców_warszawy,Parking Pl. Powstańców Warszawy,2026-07-10 12:21:05,2026-07-10 10:21:29.327023+00:00,297,15,7,396,16,8,Pl. Powstańców Warszawy,samochód osobowy,NaN,52.234989,4,21.013279,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,parking_pod_placem_krasińskich,PARKING POD PLACEM KRASIŃSKICH,2026-07-10 12:21:05,2026-07-10 10:21:29.327023+00:00,96,0,0,356,4,0,NaN,"samochód osobowy, motor",Publiczny,52.249166,1,21.004752,City Parking Group S.A.,parking.krasinskich@citypg.pl,+48 785 604 276,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,+48 785 604 276,"Gotówka, karta płatnicza",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,parking_pod_ulicą_waryńskiego,PARKING POD ULICĄ WARYŃSKIEGO,2026-07-10 12:21:05,2026-07-10 10:21:29.327023+00:00,21,0,0,130,0,0,NaN,samochód osobowy,Publiczny,52.219031,1,21.015129,City Parking Group S.A.,parking.warynskiego@citypg.pl,+48 721 910 204,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,+48 721 910 204,"Gotówka, karta płatnicza",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
m.save("parking_map.html")